# Support Vector Machine (SVM) for Classification
**Objective:** Implement SVM classifiers for binary and multiclass classification and understand the kernel trick.

## What you will learn
- Split data into training and testing sets.
- Scale features before training an SVM.
- Build a **binary classifier** using the Breast Cancer dataset.
- Build a **multiclass classifier** using the Iris dataset.
- Compare linear and RBF kernels.
- Evaluate models using accuracy, confusion matrix, and classification report.

**How to use:** Upload this `.ipynb` file to Google Drive, then open it with Google Colab. Run the cells from top to bottom.

## 1. Import required libraries
We use `pandas` to display data, `scikit-learn` for datasets and SVM models, and `matplotlib`/`seaborn` for simple visualizations. These libraries are available in Google Colab.

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Binary classification: Breast Cancer dataset
This dataset has two classes: **malignant** and **benign**. We use a subset of measurements to keep the example easy to follow.

**Important:** SVM is sensitive to feature scales, so we standardize the features. The scaler is fitted only on the training data to avoid data leakage.

In [ ]:
# Load the built-in Breast Cancer dataset
cancer = load_breast_cancer()

# Select the first 5 features to keep this practical simple
X_binary = pd.DataFrame(cancer.data[:, :5], columns=cancer.feature_names[:5])
y_binary = cancer.target  # 0 = malignant, 1 = benign

print("First five rows of features:")
display(X_binary.head())
print("Target classes:", cancer.target_names)


In [ ]:
# Divide data into training (80%) and testing (20%) parts
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_binary, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# Standardize features: mean becomes about 0 and standard deviation about 1
scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)  # Learn scaling from training data
X_test_b_scaled = scaler_b.transform(X_test_b)        # Apply same scaling to test data

print("Training samples:", len(X_train_b))
print("Testing samples:", len(X_test_b))


## 3. Train a binary SVM classifier
`kernel="linear"` uses a straight decision boundary. The model tries to find a boundary with the largest possible margin between the classes.

In [ ]:
# Create and train a linear SVM classifier
svm_binary = SVC(kernel="linear", C=1.0)
svm_binary.fit(X_train_b_scaled, y_train_b)

# Predict the class of each test example
y_pred_b = svm_binary.predict(X_test_b_scaled)

# Evaluate the model
print(f"Binary classification accuracy: {accuracy_score(y_test_b, y_pred_b):.2%}")
print("\\nClassification report:")
print(classification_report(y_test_b, y_pred_b, target_names=cancer.target_names))


In [ ]:
# Display the confusion matrix
cm_b = confusion_matrix(y_test_b, y_pred_b)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_b, annot=True, fmt="d", cmap="Blues",
    xticklabels=cancer.target_names,
    yticklabels=cancer.target_names
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Binary SVM: Confusion Matrix")
plt.show()


## 4. Multiclass classification: Iris dataset
The Iris dataset contains three flower classes: **setosa**, **versicolor**, and **virginica**. `SVC` supports multiclass classification automatically (using one-vs-one internally).

In [ ]:
# Load the Iris dataset
iris = load_iris()
X_multi = pd.DataFrame(iris.data, columns=iris.feature_names)
y_multi = iris.target

display(X_multi.head())
print("Classes:", iris.target_names)


In [ ]:
# Split into training and testing data
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

# Scale the features
scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled = scaler_m.transform(X_test_m)


## 5. Train a multiclass SVM using the RBF kernel
The **RBF (Radial Basis Function) kernel** can create curved, non-linear decision boundaries. The kernel trick lets SVM model complex class boundaries without explicitly calculating coordinates in a higher-dimensional space.

In [ ]:
# Create and train an SVM with the RBF kernel
svm_multi = SVC(kernel="rbf", C=1.0, gamma="scale")
svm_multi.fit(X_train_m_scaled, y_train_m)

# Predict flower classes
y_pred_m = svm_multi.predict(X_test_m_scaled)

print(f"Multiclass classification accuracy: {accuracy_score(y_test_m, y_pred_m):.2%}")
print("\\nClassification report:")
print(classification_report(y_test_m, y_pred_m, target_names=iris.target_names))


In [ ]:
# Show the multiclass confusion matrix
cm_m = confusion_matrix(y_test_m, y_pred_m)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_m, annot=True, fmt="d", cmap="Greens",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names
)
plt.xlabel("Predicted flower")
plt.ylabel("Actual flower")
plt.title("Multiclass SVM (RBF): Confusion Matrix")
plt.show()


## 6. Compare different kernels on the Iris dataset
- **Linear:** straight decision boundaries.
- **RBF:** flexible, curved decision boundaries.
- **Polynomial:** can model curved boundaries using polynomial relationships.

We use the same train/test split and scaled data for a fair, simple comparison.

In [ ]:
# Train and compare three kernel choices
kernel_results = []

for kernel_name in ["linear", "rbf", "poly"]:
    model = SVC(kernel=kernel_name, C=1.0, gamma="scale")
    model.fit(X_train_m_scaled, y_train_m)
    predictions = model.predict(X_test_m_scaled)
    score = accuracy_score(y_test_m, predictions)

    kernel_results.append({
        "Kernel": kernel_name,
        "Test accuracy": round(score, 4)
    })

kernel_results_df = pd.DataFrame(kernel_results)
display(kernel_results_df)


## 7. Understand support vectors
**Support vectors** are training examples closest to the decision boundary. They help determine the position of the boundary and its margin.

In [ ]:
# Number of support vectors found for each class in the Iris model
print("Support vectors per class:", svm_multi.n_support_)
print("Total support vectors:", len(svm_multi.support_))


## 8. Key takeaways
1. SVM finds a decision boundary that separates classes with a large margin.
2. Feature scaling is important for SVM.
3. A linear kernel is useful when a straight boundary is appropriate.
4. The RBF kernel can model non-linear patterns using the kernel trick.
5. `C` controls the trade-off between a wider margin and training errors: a larger `C` penalizes training errors more strongly.
6. `gamma` controls how local the influence of individual training examples is for RBF and other applicable kernels.
7. Accuracy is useful, but also inspect the confusion matrix and precision/recall report.

### Practical exercise
1. Change `kernel="linear"` to `kernel="rbf"` in the binary classifier and compare accuracy.
2. Try `C=0.1`, `C=1`, and `C=10`.
3. Record the accuracy for each setting and write two observations.

**Note:** Results can vary with dataset and settings. This practical uses built-in sample datasets, so no separate dataset upload is required.